# Module 05 — Lesson 04
# Selecting Rows and Columns, Filtering

---

> Lesson 03 got a real dataset onto disk and into a DataFrame. But a full table of 244 rows is rarely the answer to your question -- you usually want "just the dinner bills," or "just the big parties," or "just two of the seven columns." This lesson is how you ask a DataFrame those questions directly, using **boolean filtering** and `.loc`.

We'll keep using `tips.csv` from Lesson 03 -- the restaurant bills dataset -- copied into this lesson's own `data/` folder.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Filter rows with a single boolean condition (`df[df['col'] > x]`)
- Combine multiple conditions correctly using `&` / `|` with parentheses (not Python's `and`/`or`)
- Use `.isin()` for membership checks and `.between()` for range checks
- Filter rows AND select columns in one step with `.loc[row_condition, column_list]`
- Recognize and avoid the two most common filtering bugs: operator precedence and chained indexing

In [6]:
import pandas as pd

tips = pd.read_csv("data/tips.csv")
print(f"tips.shape: {tips.shape}")
print(tips.head())

tips.shape: (244, 7)
   total_bill   tip     sex smoker  day    time  size
0       16.99  1.01  Female     No  Sun  Dinner     2
1       10.34  1.66    Male     No  Sun  Dinner     3
2       21.01  3.50    Male     No  Sun  Dinner     3
3       23.68  3.31    Male     No  Sun  Dinner     2
4       24.59  3.61  Female     No  Sun  Dinner     4


---
## 1. Boolean Filtering — The Basic Pattern

`df[condition]` where `condition` is a boolean Series (one `True`/`False` per row) keeps only the rows where it's `True`. Under the hood, `tips['time'] == 'Dinner'` produces that boolean Series first -- then the outer brackets apply it.

In [7]:
is_dinner = tips["time"] == "Dinner"
print("The condition itself is a boolean Series:")
print(is_dinner.head())
print()

dinner_only = tips[is_dinner]
print(f"tips[is_dinner] keeps only True rows -> shape {dinner_only.shape}")
print(dinner_only.head(3))

The condition itself is a boolean Series:
0    True
1    True
2    True
3    True
4    True
Name: time, dtype: bool

tips[is_dinner] keeps only True rows -> shape (176, 7)
   total_bill   tip     sex smoker  day    time  size
0       16.99  1.01  Female     No  Sun  Dinner     2
1       10.34  1.66    Male     No  Sun  Dinner     3
2       21.01  3.50    Male     No  Sun  Dinner     3


In [8]:
# You don't need the intermediate variable -- this one-liner is the common form
big_bills = tips[tips["total_bill"] > 30]
print(f"Bills over $30: {big_bills.shape[0]} rows")
print(big_bills.head(3))

Bills over $30: 32 rows
    total_bill   tip     sex smoker  day    time  size
11       35.26  5.00  Female     No  Sun  Dinner     4
23       39.42  7.58    Male     No  Sat  Dinner     4
39       31.27  5.00    Male     No  Sat  Dinner     3


---
## 2. Combining Multiple Conditions

Use `&` (and), `|` (or), and `~` (not) -- **never** Python's `and`/`or`/`not`, which don't work element-by-element on a Series. Every individual condition needs its own parentheses, because `&` and `|` bind *tighter* than comparison operators like `==` and `>`.

In [9]:
# AND: both conditions must be True -- note the parentheses around EACH condition
no_smoke_dinner = tips[(tips["smoker"] == "No") & (tips["time"] == "Dinner")]
print(f"Non-smokers at dinner: {no_smoke_dinner.shape[0]} rows")
print()

# OR: either condition can be True
weekend = tips[(tips["day"] == "Sat") | (tips["day"] == "Sun")]
print(f"Saturday or Sunday: {weekend.shape[0]} rows")

Non-smokers at dinner: 106 rows

Saturday or Sunday: 163 rows


---
## 3. `.isin()` — Membership in a List

When you're checking one column against several possible values, `.isin()` reads far better than a chain of `|` comparisons -- `tips['day'].isin(['Sat', 'Sun'])` says exactly what it means.

In [10]:
weekend_isin = tips[tips["day"].isin(["Sat", "Sun"])]
print(f".isin(['Sat', 'Sun']): {weekend_isin.shape[0]} rows")
print(f"Same result as the OR version above: {weekend_isin.shape[0] == weekend.shape[0]}")

.isin(['Sat', 'Sun']): 163 rows
Same result as the OR version above: True


---
## 4. `.between()` — Filtering a Numeric Range

`.between(low, high)` is shorthand for `(col >= low) & (col <= high)` -- inclusive on both ends by default.

In [11]:
mid_range_bills = tips[tips["total_bill"].between(10, 20)]
print(f"Bills between $10 and $20: {mid_range_bills.shape[0]} rows")
print(mid_range_bills.head(3))

Bills between $10 and $20: 130 rows
   total_bill   tip     sex smoker  day    time  size
0       16.99  1.01  Female     No  Sun  Dinner     2
1       10.34  1.66    Male     No  Sun  Dinner     3
8       15.04  1.96    Male     No  Sun  Dinner     2


---
## 5. Filtering Rows AND Selecting Columns Together: `.loc`

`df[condition]` alone still gives you **every column**. To filter rows and pick specific columns in one step -- the safest and most common pattern -- pass both to `.loc`.

In [12]:
# .loc[row_condition, column_list] -- filter AND select in a single lookup
dinner_bills_only = tips.loc[tips["time"] == "Dinner", ["total_bill", "tip"]]
print("Dinner rows, only 'total_bill' and 'tip' columns:")
print(dinner_bills_only.head())

Dinner rows, only 'total_bill' and 'tip' columns:
   total_bill   tip
0       16.99  1.01
1       10.34  1.66
2       21.01  3.50
3       23.68  3.31
4       24.59  3.61


---
## 6. `.query()` — A Readable Alternative

For more complex conditions, `.query()` lets you write the filter as a plain string expression -- no repeated `df[...]` or `&`/`|` needed. It's optional, but many people find it easier to read once conditions stack up.

In [13]:
via_boolean = tips[(tips["total_bill"] > 30) & (tips["size"] <= 2)]
via_query = tips.query("total_bill > 30 and size <= 2")

print(f"Boolean indexing shape: {via_boolean.shape}")
print(f"query() shape:          {via_query.shape}")
print(f"Results match: {via_boolean.reset_index(drop=True).equals(via_query.reset_index(drop=True))}")

Boolean indexing shape: (6, 7)
query() shape:          (6, 7)
Results match: True


---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Using Python's `and`/`or` instead of `&`/`|` ---------------------

print("Mistake 1 — Python's `and` doesn't work element-by-element on a Series:")
try:
    bad = tips[tips["smoker"] == "No" and tips["time"] == "Dinner"]
except ValueError as e:
    print(f"  Caught error: {e}")
print()
print("  Correct: use `&` with each condition in its own parentheses:")
print(tips[(tips["smoker"] == "No") & (tips["time"] == "Dinner")].shape)

In [ ]:
# -- Mistake 2: Forgetting parentheses around each condition --------------------

print("Mistake 2 — `&` binds tighter than `==`, so skipping parentheses breaks:")
try:
    bad = tips[tips["smoker"] == "No" & tips["time"] == "Dinner"]
except TypeError as e:
    print(f"  Caught error: {e}")
print()
print("  Correct: wrap EVERY condition in its own parentheses:")
print(tips[(tips["smoker"] == "No") & (tips["time"] == "Dinner")].shape)

In [ ]:
# -- Mistake 3: Chained indexing when assigning to a filtered result ------------

print("Mistake 3 — filtering then assigning through a SECOND bracket lookup is unsafe:")
tips_copy = tips.copy()
print("  tips_copy[tips_copy['size'] > 4]['tip'] = 0   # DON'T -- may silently fail")
print("  to modify tips_copy (pandas can't guarantee the filter didn't return a copy).")
print()
print("  Correct: use .loc with the row condition AND column in ONE lookup:")
tips_copy.loc[tips_copy["size"] > 4, "tip"] = 0
print(tips_copy[tips_copy["size"] > 4].head(3))

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Single Condition

Filter `tips` to keep only rows where `"size"` is 4 or more. Print the resulting `.shape` and `.head()`.

In [ ]:
# Your code here

### Exercise 2 — `.isin()`

Filter `tips` to keep only rows where `"day"` is `"Thur"` or `"Fri"`, using `.isin()`. Print how many rows match.

In [ ]:
# Your code here

### Exercise 3 — `.between()` + `.loc`

Using `.loc`, filter rows where `"total_bill"` is between 15 and 25 (inclusive), and select only the `"total_bill"` and `"day"` columns. Print the first 5 rows of the result.

In [ ]:
# Your code here

### Exercise 4 — Combining Conditions

Filter `tips` to keep rows where `"time"` is `"Lunch"` **and** `"tip"` is greater than 3. Print the resulting `.shape`. (Remember: parentheses around each condition!)

In [ ]:
# Your code here

### Exercise 5 — (Challenge) `.query()` vs Boolean Indexing

Write the same filter two ways: (a) boolean indexing with `&`, and (b) `.query()`. The filter: `"sex"` is `"Female"` and `"smoker"` is `"Yes"`. Confirm both give the same shape.

In [ ]:
# Your code here

---
## 📌 Key Takeaways

- **Boolean indexing (`df[condition]`) is the core pattern for filtering rows** — combine multiple conditions with `&` / `|` / `~`, always wrapping each individual condition in parentheses, and never use Python's `and`/`or`/`not` on a Series.

- **`.loc[row_condition, column_list]` filters rows and selects columns in one step** — this is the safe way to both filter and assign, avoiding the chained-indexing trap that can silently fail to update your data.

- **`.isin()` and `.between()` read better than long comparison chains** for membership and range checks — reach for them whenever you're comparing one column against several values or a numeric range.

---
## What's Next?

**Lesson 05 — Sorting Data** takes the filtered results from this lesson and orders them — "show me the top 5 biggest bills," "show me parties sorted by day and then by size."